completion rate = users who reach confirm/total users

Hypothesis
H0 (null): No difference in completion rate between Control and Test
H1 (alternative): Test group has higher completion rate

Use:Chi-square test (best for proportions)

Output to report:
test statistic
p-value
conclusion (reject / fail to reject H0)

In [20]:
import pandas as pd
from Clean_Data import get_cleaned_vanguard_data

In [21]:
df = get_cleaned_vanguard_data()

In [19]:
df

,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,visit_id,process_step,date_time,Variation
0,836976,6.0,73.0,60.5,Unknown,2.0,45105.30,6.0,9.0,228976764_46825473280_96584,confirm,2017-04-02 11:51:13,Test
1,836976,6.0,73.0,60.5,Unknown,2.0,45105.30,6.0,9.0,228976764_46825473280_96584,confirm,2017-04-02 11:47:50,Test
2,836976,6.0,73.0,60.5,Unknown,2.0,45105.30,6.0,9.0,228976764_46825473280_96584,confirm,2017-04-02 11:46:45,Test
3,836976,6.0,73.0,60.5,Unknown,2.0,45105.30,6.0,9.0,228976764_46825473280_96584,step_3,2017-04-02 11:23:08,Test
4,836976,6.0,73.0,60.5,Unknown,2.0,45105.30,6.0,9.0,228976764_46825473280_96584,step_2,2017-04-02 11:22:24,Test
...,...,...,...,...,...,...,...,...,...,...,...,...,...
317118,7468138,18.0,222.0,61.0,F,3.0,209278.15,0.0,3.0,769876461_30381166055_830233,step_2,2017-03-30 23:59:15,Test
317119,7468138,18.0,222.0,61.0,F,3.0,209278.15,0.0,3.0,769876461_30381166055_830233,step_1,2017-03-30 23:58:51,Test
317120,7468138,18.0,222.0,61.0,F,3.0,209278.15,0.0,3.0,769876461_30381166055_830233,start,2017-03-30 23:58:40,Test
317121,7468138,18.0,222.0,61.0,F,3.0,209278.15,0.0,3.0,769876461_30381166055_830233,start,2017-03-30 23:55:11,Test


In [4]:
df.shape

(317123, 13)

In [22]:
import pandas as pd

def create_time_aware_behavior_report(df):
    
    # 1. Sort chronologically so we see the journey exactly as it happened.
    df['date_time'] = pd.to_datetime(df['date_time'])
    df = df.sort_values(by=['client_id', 'date_time'])
    
    # Mapping steps to numbers to detect forward/backward movement
    step_map = {'start': 0, 'step_1': 1, 'step_2': 2, 'step_3': 3, 'confirm': 4}
    ideal_sequence = ['start', 'step_1', 'step_2', 'step_3', 'confirm']

    def classify_journey(group):
        steps = group['process_step'].tolist()
        step_nums = [step_map.get(s, -1) for s in steps]
        
        if steps == ideal_sequence:
            return 'perfect_path'
        
        for i in range(1, len(step_nums)):
            if step_nums[i] < step_nums[i-1]:
                return 'confused'
        
        if 'confirm' in steps:
            for i in range(1, len(step_nums)):
                if step_nums[i] > step_nums[i-1] + 1:
                    return 'skipped_steps'
            
            if steps[0] != 'start':
                return 'skipped_steps'
                
            return 'successful_with_repeats'
            
        return 'dropped_in_between'

    # 2. Apply the logic to each unique visit
    print("Analyzing chronological journeys...")
    # Using .apply(include_groups=False) if you are on a newer pandas version to avoid warnings
    classified_clients = df.groupby(['client_id']).apply(classify_journey).reset_index()
    classified_clients.columns = ['client_id', 'path_category']

    # 3. Priority Aggregation
    priority = {
        'perfect_path': 1,
        'successful_with_repeats': 2,
        'skipped_steps': 3,
        'confused': 4,
        'dropped_in_between': 5
    }
    classified_clients['priority'] = classified_clients['path_category'].map(priority)
    
    final_behavior = (classified_clients.sort_values('priority')
                .groupby('client_id')
                .first()
                .reset_index())

    # ✅ NEW: Extract Demographic & Experiment info per client
    # We take the 'first' non-null value for each attribute
    client_info = (
        df.groupby('client_id')
        .agg({
            'Variation': 'first',
            'clnt_age': 'first',
            'gendr':'first',
            'clnt_tenure_yr': 'first',
            'bal': 'first'
        })
        .reset_index()
    )

    # 4. Merge behavior with client info
    client_level_df = final_behavior.merge(client_info, on='client_id', how='left')

    # Drop rows where Variation is still None after the merge
    client_level_df = client_level_df.dropna(subset=['Variation'])
    
    return client_level_df[['client_id', 'path_category', 'Variation', 'clnt_age', 'gendr','clnt_tenure_yr', 'bal']]


# Example usage:
client_level_df = create_time_aware_behavior_report(df)

Analyzing chronological journeys...


C:\Users\advik\AppData\Local\Temp\ipykernel_5228\504504374.py:39: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  classified_clients = df.groupby(['client_id']).apply(classify_journey).reset_index()


In [23]:
client_level_df

,client_id,path_category,Variation,clnt_age,gendr,clnt_tenure_yr,bal
0,555,perfect_path,Test,29.5,Unknown,3.0,25454.66
1,647,perfect_path,Test,57.5,M,12.0,30525.80
2,934,dropped_in_between,Test,51.0,F,9.0,32522.88
3,1028,confused,Control,36.0,M,12.0,103520.22
4,1104,dropped_in_between,Control,48.0,Unknown,5.0,154643.94
...,...,...,...,...,...,...,...
50482,9999150,confused,Test,30.0,Unknown,5.0,97141.71
50483,9999400,perfect_path,Test,28.5,Unknown,7.0,51787.04
50484,9999626,dropped_in_between,Test,35.0,M,9.0,36642.88
50485,9999729,confused,Test,31.0,F,10.0,107059.74


Z-test: Tells you which group is better. Because the Z-statistic can be positive or negative, it immediately shows if the Test group "lifted" or "dropped" the conversion rate.

Chi-square: Only tells you if there is a difference, not the direction. A Chi-square statistic is always positive, so it can't tell you on its own if the new design was better or worse than the old one—just that they aren't the same.

In [28]:
# Select both successful categories and sum them up for the 'Test' group
counts = client_level_df.groupby(['Variation', 'path_category']).size().unstack(fill_value=0)

success_test = counts.loc['Test', ['perfect_path', 'successful_with_repeats']].sum()
total_test = counts.loc['Test'].sum()

# Do the same for the 'Control' group
success_control = counts.loc['Control', ['perfect_path', 'successful_with_repeats']].sum()
total_control = counts.loc['Control'].sum()

# Then run your Z-test using these new 'success' numbers
from statsmodels.stats.proportion import proportions_ztest
z_stat, p_value = proportions_ztest([success_test, success_control], [total_test, total_control])
z_stat, p_value

(np.float64(8.556957185289672), np.float64(1.1588554767718645e-17))

In [17]:
counts

path_category,confused,dropped_in_between,perfect_path,skipped_steps,successful_with_repeats
Variation,,,,,
Control,8074,5542,7358,172,2380
Test,10189,4290,8271,302,3909


In [29]:
# Calculate the proportions
p_test = success_test / total_test
p_control = success_control / total_control

# Calculate Lift
lift = (p_test - p_control) / p_control

print(f"Test Proportion: {p_test:.4f}")
print(f"Control Proportion: {p_control:.4f}")
print(f"Lift: {lift:.2%}")

Test Proportion: 0.4518
Control Proportion: 0.4139
Lift: 9.14%


In [26]:
# client who reached to confirm
completed_users = client_level_df[client_level_df["path_category"] == "perfect_path"]["client_id"].nunique()

# ttal users per group
total_users = client_level_df["client_id"].nunique()

# users who completed
completed = client_level_df[client_level_df["path_category"] == "perfect_path"].groupby("Variation")["client_id"].nunique()

# total users per group
total = client_level_df.groupby("Variation")["client_id"].nunique()

# completion rate
completion_rate = completed / total

print(completion_rate)

Variation
Control    0.312760
Test       0.306776
Name: client_id, dtype: float64


Perfom Chi-square Test

In [27]:
from scipy.stats import chi2_contingency
import numpy as np

# 1. Prepare the success and failure counts
# Success = Perfect + Repeats
success_test = counts.loc['Test', ['perfect_path', 'successful_with_repeats']].sum()
fail_test = counts.loc['Test'].sum() - success_test

success_control = counts.loc['Control', ['perfect_path', 'successful_with_repeats']].sum()
fail_control = counts.loc['Control'].sum() - success_control

# 2. Create the Contingency Table
observed = np.array([
    [success_test, fail_test],
    [success_control, fail_control]
])

# 3. Execute the Chi-Square Test
chi2_stat, p_val, dof, expected = chi2_contingency(observed)

print(f"Chi-Square Statistic: {chi2_stat}")
print(f"P-value: {p_val}")

Chi-Square Statistic: 73.06756962629068
P-value: 1.252863265388749e-17


In [52]:
from scipy import stats

age_test = client_level_df[client_level_df['Variation'] == 'Test']['clnt_age'].dropna()
age_control = client_level_df[client_level_df['Variation'] == 'Control']['clnt_age'].dropna()
# 1. Chi-Square (Success/Failure)
chi2, p_chi, dof, expected = stats.chi2_contingency(observed)

# 2. T-Test (Comparing Age)
t_stat, p_t = stats.ttest_ind(age_test, age_control)

print(f"Chi-Square Result: {p_chi}")
print(f"T-Test Result: {p_t}")

Chi-Square Result: 1.252863265388749e-17
T-Test Result: 0.0156893118497501


You have evidence that the "Test" group and "Control" group did not perform the same.